# Các file cần thiết: 
 1. LST NRDUCELL.xlsx
 2. LST NRDUCELLTRP.xlsx
 3. LST NRDUCELLCOVERAGE.xlsx
 4. Inventory_Board.csv
 5. LST GNODEBFUNCTION.xlsx

In [56]:
import pandas as pd

In [57]:
def read_excel_columns(file_path, columns, header_row=1, rename_dict=None):
    """
    Đọc file Excel, lấy các cột cần thiết, đổi tên cột nếu cần.
    file_path: đường dẫn file Excel
    columns: danh sách tên cột cần lấy
    header_row: số thứ tự dòng header (mặc định 1, tức là dòng thứ 2)
    rename_dict: dict đổi tên cột nếu cần (mặc định None)
    """
    df = pd.read_excel(file_path, header=header_row, engine='openpyxl')
    df = df[[col for col in columns if col in df.columns]]
    if rename_dict:
        df = df.rename(columns=rename_dict)
    return df

# Ví dụ sử dụng:
# df_cell = read_excel_columns('LST CELL_2025_05_27_11_34_17_832.xlsx', columns_needed)
# df_cell_sectoreq = read_excel_columns('LST EUCELLSECTOREQM_2025_05_27_11_35_12_418.xlsx', columns_eucell_needed, rename_dict={'Sector equipment ID': 'Sector Equipment ID'})

In [58]:
# Đọc file Inventory_Board_20250527_111941.csv cho BBU và RRU
file_board = 'Inventory_Board.csv'

# Các cột cho BBU
columns_bbu = ['NEName', 'Board Name', 'Board Type']
df_bbu = pd.read_csv(file_board, usecols=[col for col in columns_bbu if col])
# Lọc các dòng có chữ 'UBBP' trong cột Board Name
mask_ubbp = df_bbu['Board Name'].astype(str).str.contains('UBBP', na=False)
df_bbu = df_bbu[mask_ubbp]
# Dùng regex lấy dữ liệu bắt đầu bằng 'UBB' cho đến hết dòng trong Board Type
import re
def extract_ubb(text):
    match = re.search(r'(UBB\w*)', str(text))
    return match.group(1) if match else ''
df_bbu['Board Type'] = df_bbu['Board Type'].apply(extract_ubb)

# Các cột cho RRU (không lấy Board Type)
columns_rru = ['NEName', 'Board Name', 'Subrack No.', 'Manufacturer Data']
df_rru = pd.read_csv(file_board, usecols=[col for col in columns_rru if col])
# Lọc các dòng có chữ 'MRRU' trong cột Board Name
mask_mrru = df_rru['Board Name'].astype(str).str.contains('AIRU', na=False)
df_rru = df_rru[mask_mrru]
# Giữ lại thông tin trước dấu phẩy đầu tiên trong Manufacturer Data
if 'Manufacturer Data' in df_rru.columns:
    df_rru['Manufacturer Data'] = df_rru['Manufacturer Data'].astype(str).str.split(',').str[0]
    df_rru = df_rru.rename(columns={'Manufacturer Data': 'RF Type'})

df_bbu.head(), df_rru.head()

(    NEName Board Name Board Type
 1   AGTS33       UBBP     UBBPe2
 15  AGTB17       UBBP     UBBPd4
 27  AGTB20       UBBP     UBBPe2
 47  CMDD30       UBBP     UBBPd4
 55  CMDD35       UBBP     UBBPe2,
            NEName Board Name  Subrack No.   RF Type
 73933  CTNK67NR77       AIRU          200  AAU5656m
 73934  CTNK67NR77       AIRU          201  AAU5656m
 73936  CTNK67NR77       AIRU          202  AAU5656m
 73972  CTNK03NR77       AIRU          202  AAU5656m
 73973  CTNK03NR77       AIRU          201  AAU5656m)

In [59]:
df_bbu.head(10)

,NEName,Board Name,Board Type
1,AGTS33,UBBP,UBBPe2
15,AGTB17,UBBP,UBBPd4
27,AGTB20,UBBP,UBBPe2
47,CMDD30,UBBP,UBBPd4
55,CMDD35,UBBP,UBBPe2
81,CMDD42,UBBP,UBBPe2
96,CMTT34,UBBP,UBBPd4
111,CMNC23,UBBP,UBBPe2
113,STVC39,UBBP,UBBPe2
138,STMX21,UBBP,UBBPd4


In [60]:
df_bbu.info()
df_bbu.describe()

<class 'pandas.core.frame.DataFrame'>
Index: 8025 entries, 1 to 99324
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   NEName      8025 non-null   object
 1   Board Name  8025 non-null   object
 2   Board Type  8025 non-null   object
dtypes: object(3)
memory usage: 250.8+ KB


,NEName,Board Name,Board Type
count,8025,8025,8025
unique,5764,1,10
top,CTNKZ1NR77,UBBP,UBBPe2
freq,4,8025,1878


In [61]:

# Thêm cột đếm số lượng cho mỗi cặp NEName và Board Type
df_bbu['Count'] = df_bbu.groupby(['NEName','Board Type'])['Board Type'].transform('count')     
df_bbu.head(10)

,NEName,Board Name,Board Type,Count
1,AGTS33,UBBP,UBBPe2,1
15,AGTB17,UBBP,UBBPd4,1
27,AGTB20,UBBP,UBBPe2,1
47,CMDD30,UBBP,UBBPd4,1
55,CMDD35,UBBP,UBBPe2,1
81,CMDD42,UBBP,UBBPe2,1
96,CMTT34,UBBP,UBBPd4,1
111,CMNC23,UBBP,UBBPe2,1
113,STVC39,UBBP,UBBPe2,1
138,STMX21,UBBP,UBBPd4,1


In [62]:
df_bbu.sort_values(by=['NEName', 'Count'], inplace=True)
df_bbu.head(10)

,NEName,Board Name,Board Type,Count
67961,AGAP01,UBBP,UBBPd4,1
49315,AGAP02,UBBP,UBBPe2,1
49595,AGAP03,UBBP,UBBPe2,1
970,AGAP04,UBBP,UBBPe2,1
67971,AGAP05,UBBP,UBBPd4,1
49295,AGAP06,UBBP,UBBPe2,1
29575,AGAP07,UBBP,UBBPe2,1
32297,AGAP08,UBBP,UBBPd4,1
32305,AGAP08,UBBP,UBBPe2,1
91347,AGAP08NR77,UBBP,UBBPg3a,1


In [63]:
df_bbu['Config'] = df_bbu['Count'].astype(str) + '_' + df_bbu['Board Type'].astype(str)
df_bbu.head()

,NEName,Board Name,Board Type,Count,Config
67961,AGAP01,UBBP,UBBPd4,1,1_UBBPd4
49315,AGAP02,UBBP,UBBPe2,1,1_UBBPe2
49595,AGAP03,UBBP,UBBPe2,1,1_UBBPe2
970,AGAP04,UBBP,UBBPe2,1,1_UBBPe2
67971,AGAP05,UBBP,UBBPd4,1,1_UBBPd4


In [64]:
df_bbu = df_bbu[['NEName','Config']].reset_index(drop=True)
df_bbu = df_bbu.drop_duplicates()
df_bbu.head(10)

,NEName,Config
0,AGAP01,1_UBBPd4
1,AGAP02,1_UBBPe2
2,AGAP03,1_UBBPe2
3,AGAP04,1_UBBPe2
4,AGAP05,1_UBBPd4
5,AGAP06,1_UBBPe2
6,AGAP07,1_UBBPe2
7,AGAP08,1_UBBPd4
8,AGAP08,1_UBBPe2
9,AGAP08NR77,1_UBBPg3a


In [65]:
#df["Danh sách điểm"] = (df.groupby("Họ tên")["Điểm"].transform(lambda x: ", ".join(map(str, x))))
df_bbu['BBU'] = (df_bbu.groupby("NEName")["Config"].transform(lambda x: "+".join(map(str, x))))
df_bbu.head(10)

,NEName,Config,BBU
0,AGAP01,1_UBBPd4,1_UBBPd4
1,AGAP02,1_UBBPe2,1_UBBPe2
2,AGAP03,1_UBBPe2,1_UBBPe2
3,AGAP04,1_UBBPe2,1_UBBPe2
4,AGAP05,1_UBBPd4,1_UBBPd4
5,AGAP06,1_UBBPe2,1_UBBPe2
6,AGAP07,1_UBBPe2,1_UBBPe2
7,AGAP08,1_UBBPd4,1_UBBPd4+1_UBBPe2
8,AGAP08,1_UBBPe2,1_UBBPd4+1_UBBPe2
9,AGAP08NR77,1_UBBPg3a,1_UBBPg3a


In [66]:
df_bbu = df_bbu[['NEName','BBU']].reset_index(drop=True)
df_bbu = df_bbu.drop_duplicates()
df_bbu.head(10)

,NEName,BBU
0,AGAP01,1_UBBPd4
1,AGAP02,1_UBBPe2
2,AGAP03,1_UBBPe2
3,AGAP04,1_UBBPe2
4,AGAP05,1_UBBPd4
5,AGAP06,1_UBBPe2
6,AGAP07,1_UBBPe2
7,AGAP08,1_UBBPd4+1_UBBPe2
9,AGAP08NR77,1_UBBPg3a
10,AGAP09,1_UBBPe2


In [67]:
df_rru.head(10)

,NEName,Board Name,Subrack No.,RF Type
73933,CTNK67NR77,AIRU,200,AAU5656m
73934,CTNK67NR77,AIRU,201,AAU5656m
73936,CTNK67NR77,AIRU,202,AAU5656m
73972,CTNK03NR77,AIRU,202,AAU5656m
73973,CTNK03NR77,AIRU,201,AAU5656m
73974,CTNK03NR77,AIRU,200,AAU5656m
73977,CTNKZ0NR77,AIRU,202,AAU5656m
73978,CTNKZ0NR77,AIRU,201,AAU5656m
73979,CTNKZ0NR77,AIRU,200,AAU5656m
74011,CTNK12NR77,AIRU,202,AAU5656m


In [ ]:
# đọc file LST SECTOREQM.xlsx
columns_sectoreq = [
    'Base Station Name', 'Sector Equipment ID', 'RRU Subrack No.'
]
file_sectoreq 'LST SECTOREQM.xlsx'
df_sectoreq = read_excel_columns(file_sectoreq, columns_sectoreq)
df_sectoreq = df_sectoreq.dropna()

c:\Users\Chinhnq\AppData\Local\Programs\Python\Python313\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [69]:
df_sectoreq.head(10)

,Base Station Name,Sector Equipment ID,RRU Subrack No.
0,HGVT36NR77,203,202.0
1,HGVT36NR77,202,201.0
2,HGVT36NR77,201,200.0
3,KGRG02NR77,203,202.0
4,KGRG02NR77,202,201.0
5,KGRG02NR77,201,200.0
6,CTCRL2NR77,203,202.0
7,CTCRL2NR77,202,201.0
8,CTCRL2NR77,201,200.0
9,KGRGL1NR77,203,202.0


In [82]:
# đọc flle LST NRDUCELL.xlsx
columns_nrcell = ['Base Station Name','NR DU Cell ID','NR DU Cell Name',
                'Physical Cell ID','Frequency Band','Downlink NARFCN','Downlink Bandwidth',
                'Subcarrier Spacing(KHz)','Slot Assignment','Tracking Area ID','SSB Frequency Position',
                'Logical Root Sequence Index']
file_nrcell = 'LST NRDUCELL.xlsx'
df_nrcell = read_excel_columns(file_nrcell, columns_nrcell)
df_nrcell = df_nrcell.dropna()
df_nrcell.head(10)
                

c:\Users\Chinhnq\AppData\Local\Programs\Python\Python313\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,Base Station Name,NR DU Cell ID,NR DU Cell Name,Physical Cell ID,Frequency Band,Downlink NARFCN,Downlink Bandwidth,Subcarrier Spacing(KHz),Slot Assignment,Tracking Area ID,SSB Frequency Position,Logical Root Sequence Index
0,CTCR15NR77,71.0,CTCR15M5SA,142.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,46202.0,8088.0,528.0
1,CTCR15NR77,72.0,CTCR15M5SB,143.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,46202.0,8088.0,504.0
2,CTCR15NR77,73.0,CTCR15M5SC,141.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,46202.0,8088.0,354.0
3,HGCA14NR77,71.0,HGCA14M5SA,55.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,47522.0,8088.0,102.0
4,HGCA14NR77,72.0,HGCA14M5SB,54.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,47522.0,8088.0,6.0
5,HGCA14NR77,73.0,HGCA14M5SC,56.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,47522.0,8088.0,540.0
6,CTCR08NR77,71.0,CTCR08M5SA,258.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,46202.0,8088.0,672.0
7,CTCR08NR77,72.0,CTCR08M5SB,259.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,46202.0,8088.0,168.0
8,CTCR08NR77,73.0,CTCR08M5SC,260.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,46202.0,8088.0,258.0
9,HGCA01NR77,71.0,HGCA01M5SA,217.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,47522.0,8088.0,510.0


In [71]:
# đọc file LST NRDUCELLTRP.xlsx
columns_nrcelltrp = ['Base Station Name','NR DU Cell ID','NR DU Cell TRP ID','Transmit and Receive Mode']
file_nrcelltrp = 'LST NRDUCELLTRP.xlsx'
df_nrcelltrp = read_excel_columns(file_nrcelltrp, columns_nrcelltrp)
df_nrcelltrp.head(10)

c:\Users\Chinhnq\AppData\Local\Programs\Python\Python313\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,Base Station Name,NR DU Cell ID,NR DU Cell TRP ID,Transmit and Receive Mode
0,CTTL11NR77,71.0,71.0,32T32R
1,CTTL11NR77,72.0,72.0,32T32R
2,CTTL11NR77,73.0,73.0,32T32R
3,HGVT25NR77,71.0,71.0,64T64R
4,HGVT25NR77,72.0,72.0,64T64R
5,HGVT25NR77,73.0,73.0,64T64R
6,CTTL09NR77,71.0,71.0,32T32R
7,CTTL09NR77,72.0,72.0,32T32R
8,CTTL09NR77,73.0,73.0,32T32R
9,CTCR57NR77,71.0,71.0,32T32R


In [72]:
# đọc file LST NRDUCELLCOVERAGE.xlsx
columns_nrcellcoverage = ['Base Station Name','NR DU Cell TRP ID','Sector Equipment ID']
file_nrcellcoverage = 'LST NRDUCELLCOVERAGE.xlsx'
df_nrcellcoverage = read_excel_columns(file_nrcellcoverage, columns_nrcellcoverage)
df_nrcellcoverage.head(10)

c:\Users\Chinhnq\AppData\Local\Programs\Python\Python313\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,Base Station Name,NR DU Cell TRP ID,Sector Equipment ID
0,HGLMX0NR77,71.0,201.0
1,HGLMX0NR77,72.0,202.0
2,HGLMX0NR77,73.0,203.0
3,CTTN02NR77,71.0,201.0
4,CTTN02NR77,72.0,202.0
5,CTTN02NR77,73.0,203.0
6,HGVT42NR77,71.0,201.0
7,HGVT42NR77,72.0,202.0
8,HGVT42NR77,73.0,203.0
9,HGCA49NR77,71.0,201.0


In [83]:
df_final = pd.merge(df_nrcell, df_nrcelltrp, on=['Base Station Name','NR DU Cell ID'], how='left')
df_final = pd.merge(df_final, df_nrcellcoverage, on=['Base Station Name','NR DU Cell TRP ID'], how='left')
df_final = pd.merge(df_final, df_sectoreq, on=['Base Station Name','Sector Equipment ID'], how='left')
df_final.head(10)

,Base Station Name,NR DU Cell ID,NR DU Cell Name,Physical Cell ID,Frequency Band,Downlink NARFCN,Downlink Bandwidth,Subcarrier Spacing(KHz),Slot Assignment,Tracking Area ID,SSB Frequency Position,Logical Root Sequence Index,NR DU Cell TRP ID,Transmit and Receive Mode,Sector Equipment ID,RRU Subrack No.
0,CTCR15NR77,71.0,CTCR15M5SA,142.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,46202.0,8088.0,528.0,71.0,64T64R,201.0,200.0
1,CTCR15NR77,72.0,CTCR15M5SB,143.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,46202.0,8088.0,504.0,72.0,64T64R,202.0,201.0
2,CTCR15NR77,73.0,CTCR15M5SC,141.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,46202.0,8088.0,354.0,73.0,64T64R,203.0,202.0
3,HGCA14NR77,71.0,HGCA14M5SA,55.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,47522.0,8088.0,102.0,71.0,64T64R,201.0,200.0
4,HGCA14NR77,72.0,HGCA14M5SB,54.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,47522.0,8088.0,6.0,72.0,64T64R,202.0,201.0
5,HGCA14NR77,73.0,HGCA14M5SC,56.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,47522.0,8088.0,540.0,73.0,64T64R,203.0,202.0
6,CTCR08NR77,71.0,CTCR08M5SA,258.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,46202.0,8088.0,672.0,71.0,64T64R,201.0,200.0
7,CTCR08NR77,72.0,CTCR08M5SB,259.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,46202.0,8088.0,168.0,72.0,64T64R,202.0,201.0
8,CTCR08NR77,73.0,CTCR08M5SC,260.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,46202.0,8088.0,258.0,73.0,64T64R,203.0,202.0
9,HGCA01NR77,71.0,HGCA01M5SA,217.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,47522.0,8088.0,510.0,71.0,64T64R,201.0,200.0


In [84]:
df_final = pd.merge(df_final, df_bbu, left_on='Base Station Name', right_on='NEName', how='left')
df_final = df_final.drop(columns=['NEName'])
df_final.head(10)

,Base Station Name,NR DU Cell ID,NR DU Cell Name,Physical Cell ID,Frequency Band,Downlink NARFCN,Downlink Bandwidth,Subcarrier Spacing(KHz),Slot Assignment,Tracking Area ID,SSB Frequency Position,Logical Root Sequence Index,NR DU Cell TRP ID,Transmit and Receive Mode,Sector Equipment ID,RRU Subrack No.,BBU
0,CTCR15NR77,71.0,CTCR15M5SA,142.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,46202.0,8088.0,528.0,71.0,64T64R,201.0,200.0,1_UBBPg3a
1,CTCR15NR77,72.0,CTCR15M5SB,143.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,46202.0,8088.0,504.0,72.0,64T64R,202.0,201.0,1_UBBPg3a
2,CTCR15NR77,73.0,CTCR15M5SC,141.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,46202.0,8088.0,354.0,73.0,64T64R,203.0,202.0,1_UBBPg3a
3,HGCA14NR77,71.0,HGCA14M5SA,55.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,47522.0,8088.0,102.0,71.0,64T64R,201.0,200.0,1_UBBPg3a
4,HGCA14NR77,72.0,HGCA14M5SB,54.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,47522.0,8088.0,6.0,72.0,64T64R,202.0,201.0,1_UBBPg3a
5,HGCA14NR77,73.0,HGCA14M5SC,56.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,47522.0,8088.0,540.0,73.0,64T64R,203.0,202.0,1_UBBPg3a
6,CTCR08NR77,71.0,CTCR08M5SA,258.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,46202.0,8088.0,672.0,71.0,64T64R,201.0,200.0,1_UBBPg3a
7,CTCR08NR77,72.0,CTCR08M5SB,259.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,46202.0,8088.0,168.0,72.0,64T64R,202.0,201.0,1_UBBPg3a
8,CTCR08NR77,73.0,CTCR08M5SC,260.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,46202.0,8088.0,258.0,73.0,64T64R,203.0,202.0,1_UBBPg3a
9,HGCA01NR77,71.0,HGCA01M5SA,217.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,47522.0,8088.0,510.0,71.0,64T64R,201.0,200.0,1_UBBPg3a


In [85]:
df_final = pd.merge(df_final, df_rru, left_on=['Base Station Name','RRU Subrack No.'], right_on=['NEName','Subrack No.'], how='left')
df_final = df_final.drop(columns=['NEName','Subrack No.'])
df_final.head(10)

,Base Station Name,NR DU Cell ID,NR DU Cell Name,Physical Cell ID,Frequency Band,Downlink NARFCN,Downlink Bandwidth,Subcarrier Spacing(KHz),Slot Assignment,Tracking Area ID,SSB Frequency Position,Logical Root Sequence Index,NR DU Cell TRP ID,Transmit and Receive Mode,Sector Equipment ID,RRU Subrack No.,BBU,Board Name,RF Type
0,CTCR15NR77,71.0,CTCR15M5SA,142.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,46202.0,8088.0,528.0,71.0,64T64R,201.0,200.0,1_UBBPg3a,AIRU,AAU5636v
1,CTCR15NR77,72.0,CTCR15M5SB,143.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,46202.0,8088.0,504.0,72.0,64T64R,202.0,201.0,1_UBBPg3a,AIRU,AAU5636v
2,CTCR15NR77,73.0,CTCR15M5SC,141.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,46202.0,8088.0,354.0,73.0,64T64R,203.0,202.0,1_UBBPg3a,AIRU,AAU5636v
3,HGCA14NR77,71.0,HGCA14M5SA,55.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,47522.0,8088.0,102.0,71.0,64T64R,201.0,200.0,1_UBBPg3a,AIRU,AAU5636v
4,HGCA14NR77,72.0,HGCA14M5SB,54.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,47522.0,8088.0,6.0,72.0,64T64R,202.0,201.0,1_UBBPg3a,AIRU,AAU5636v
5,HGCA14NR77,73.0,HGCA14M5SC,56.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,47522.0,8088.0,540.0,73.0,64T64R,203.0,202.0,1_UBBPg3a,AIRU,AAU5636v
6,CTCR08NR77,71.0,CTCR08M5SA,258.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,46202.0,8088.0,672.0,71.0,64T64R,201.0,200.0,1_UBBPg3a,AIRU,AAU5636v
7,CTCR08NR77,72.0,CTCR08M5SB,259.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,46202.0,8088.0,168.0,72.0,64T64R,202.0,201.0,1_UBBPg3a,AIRU,AAU5636v
8,CTCR08NR77,73.0,CTCR08M5SC,260.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,46202.0,8088.0,258.0,73.0,64T64R,203.0,202.0,1_UBBPg3a,AIRU,AAU5636v
9,HGCA01NR77,71.0,HGCA01M5SA,217.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,47522.0,8088.0,510.0,71.0,64T64R,201.0,200.0,1_UBBPg3a,AIRU,AAU5636v


In [86]:
drop_columns = ['NR DU Cell TRP ID', 'Board Name', 'RRU Subrack No.']
df_final = df_final.drop(columns=drop_columns)
df_final.head(10)

,Base Station Name,NR DU Cell ID,NR DU Cell Name,Physical Cell ID,Frequency Band,Downlink NARFCN,Downlink Bandwidth,Subcarrier Spacing(KHz),Slot Assignment,Tracking Area ID,SSB Frequency Position,Logical Root Sequence Index,Transmit and Receive Mode,Sector Equipment ID,BBU,RF Type
0,CTCR15NR77,71.0,CTCR15M5SA,142.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,46202.0,8088.0,528.0,64T64R,201.0,1_UBBPg3a,AAU5636v
1,CTCR15NR77,72.0,CTCR15M5SB,143.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,46202.0,8088.0,504.0,64T64R,202.0,1_UBBPg3a,AAU5636v
2,CTCR15NR77,73.0,CTCR15M5SC,141.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,46202.0,8088.0,354.0,64T64R,203.0,1_UBBPg3a,AAU5636v
3,HGCA14NR77,71.0,HGCA14M5SA,55.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,47522.0,8088.0,102.0,64T64R,201.0,1_UBBPg3a,AAU5636v
4,HGCA14NR77,72.0,HGCA14M5SB,54.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,47522.0,8088.0,6.0,64T64R,202.0,1_UBBPg3a,AAU5636v
5,HGCA14NR77,73.0,HGCA14M5SC,56.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,47522.0,8088.0,540.0,64T64R,203.0,1_UBBPg3a,AAU5636v
6,CTCR08NR77,71.0,CTCR08M5SA,258.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,46202.0,8088.0,672.0,64T64R,201.0,1_UBBPg3a,AAU5636v
7,CTCR08NR77,72.0,CTCR08M5SB,259.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,46202.0,8088.0,168.0,64T64R,202.0,1_UBBPg3a,AAU5636v
8,CTCR08NR77,73.0,CTCR08M5SC,260.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,46202.0,8088.0,258.0,64T64R,203.0,1_UBBPg3a,AAU5636v
9,HGCA01NR77,71.0,HGCA01M5SA,217.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,47522.0,8088.0,510.0,64T64R,201.0,1_UBBPg3a,AAU5636v


In [87]:
rename_dict = {
    'NR DU Cell Name':'Cell Name',
    'Physical Cell ID':'PCI', 'Frequency Band':'Band',
    'Downlink NARFCN':'DL NARFCN', 'Downlink Bandwidth':'DL Bandwidth',
    'Subcarrier Spacing(KHz)':'SCS (KHz)', 'Logical Root Sequence Index':'RSI',
    'Transmit and Receive Mode':'TxRx Mode',
    'Tracking Area ID':'TAC', 'SSB Frequency Position':'SSB FP'}
df_final = df_final.rename(columns=rename_dict)
df_final.head(10)


,Base Station Name,NR DU Cell ID,Cell Name,PCI,Band,DL NARFCN,DL Bandwidth,SCS (KHz),Slot Assignment,TAC,SSB FP,RSI,TxRx Mode,Sector Equipment ID,BBU,RF Type
0,CTCR15NR77,71.0,CTCR15M5SA,142.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,46202.0,8088.0,528.0,64T64R,201.0,1_UBBPg3a,AAU5636v
1,CTCR15NR77,72.0,CTCR15M5SB,143.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,46202.0,8088.0,504.0,64T64R,202.0,1_UBBPg3a,AAU5636v
2,CTCR15NR77,73.0,CTCR15M5SC,141.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,46202.0,8088.0,354.0,64T64R,203.0,1_UBBPg3a,AAU5636v
3,HGCA14NR77,71.0,HGCA14M5SA,55.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,47522.0,8088.0,102.0,64T64R,201.0,1_UBBPg3a,AAU5636v
4,HGCA14NR77,72.0,HGCA14M5SB,54.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,47522.0,8088.0,6.0,64T64R,202.0,1_UBBPg3a,AAU5636v
5,HGCA14NR77,73.0,HGCA14M5SC,56.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,47522.0,8088.0,540.0,64T64R,203.0,1_UBBPg3a,AAU5636v
6,CTCR08NR77,71.0,CTCR08M5SA,258.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,46202.0,8088.0,672.0,64T64R,201.0,1_UBBPg3a,AAU5636v
7,CTCR08NR77,72.0,CTCR08M5SB,259.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,46202.0,8088.0,168.0,64T64R,202.0,1_UBBPg3a,AAU5636v
8,CTCR08NR77,73.0,CTCR08M5SC,260.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,46202.0,8088.0,258.0,64T64R,203.0,1_UBBPg3a,AAU5636v
9,HGCA01NR77,71.0,HGCA01M5SA,217.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,47522.0,8088.0,510.0,64T64R,201.0,1_UBBPg3a,AAU5636v


In [88]:
# đọc file LST GNODEBFUNCTION.xlsx
columns_gnodebfunction = ['Base Station Name', 'gNodeB ID']
file_gnodebfunction = 'LST GNODEBFUNCTION.xlsx'
df_gnodebfunction = read_excel_columns(file_gnodebfunction, columns_gnodebfunction)
df_gnodebfunction.dropna(inplace=True)
df_gnodebfunction.head(10)

c:\Users\Chinhnq\AppData\Local\Programs\Python\Python313\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,Base Station Name,gNodeB ID
0,VLBM19NR77,7253216.0
1,DTCL28NR77,6652237.0
2,DTSD36NR77,6652420.0
3,DTHU14NR77,6652190.0
4,DTTN14NR77,6652008.0
5,VLLH35NR77,7253220.0
6,DTLV09NR77,6652044.0
7,DTLV33NR77,6652257.0
8,DTCL40NR77,6652283.0
9,DTLV03NR77,6652146.0


In [89]:
df_final = pd.merge(df_final,df_gnodebfunction, on='Base Station Name', how='left')
df_final.head(10)


,Base Station Name,NR DU Cell ID,Cell Name,PCI,Band,DL NARFCN,DL Bandwidth,SCS (KHz),Slot Assignment,TAC,SSB FP,RSI,TxRx Mode,Sector Equipment ID,BBU,RF Type,gNodeB ID
0,CTCR15NR77,71.0,CTCR15M5SA,142.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,46202.0,8088.0,528.0,64T64R,201.0,1_UBBPg3a,AAU5636v,6551093.0
1,CTCR15NR77,72.0,CTCR15M5SB,143.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,46202.0,8088.0,504.0,64T64R,202.0,1_UBBPg3a,AAU5636v,6551093.0
2,CTCR15NR77,73.0,CTCR15M5SC,141.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,46202.0,8088.0,354.0,64T64R,203.0,1_UBBPg3a,AAU5636v,6551093.0
3,HGCA14NR77,71.0,HGCA14M5SA,55.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,47522.0,8088.0,102.0,64T64R,201.0,1_UBBPg3a,AAU5636v,6751785.0
4,HGCA14NR77,72.0,HGCA14M5SB,54.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,47522.0,8088.0,6.0,64T64R,202.0,1_UBBPg3a,AAU5636v,6751785.0
5,HGCA14NR77,73.0,HGCA14M5SC,56.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,47522.0,8088.0,540.0,64T64R,203.0,1_UBBPg3a,AAU5636v,6751785.0
6,CTCR08NR77,71.0,CTCR08M5SA,258.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,46202.0,8088.0,672.0,64T64R,201.0,1_UBBPg3a,AAU5636v,6551033.0
7,CTCR08NR77,72.0,CTCR08M5SB,259.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,46202.0,8088.0,168.0,64T64R,202.0,1_UBBPg3a,AAU5636v,6551033.0
8,CTCR08NR77,73.0,CTCR08M5SC,260.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,46202.0,8088.0,258.0,64T64R,203.0,1_UBBPg3a,AAU5636v,6551033.0
9,HGCA01NR77,71.0,HGCA01M5SA,217.0,N77,656666.0,CELL_BW_100M,30KHZ,8_2_DDDDDDDSUU,47522.0,8088.0,510.0,64T64R,201.0,1_UBBPg3a,AAU5636v,6751706.0


In [91]:
df_final['Vendor'] = 'Huawei'

In [92]:
df_final.to_excel('5G_Huawei_Inventory_Output.xlsx', index=False)